In [6]:
%pip install -q openai pandas matplotlib tqdm


In [19]:
!pip -q install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.


In [22]:
from getpass import getpass
from google import genai

gemini_key = getpass("Paste your Gemini API key: ")
gemini_client = genai.Client(api_key=gemini_key)

print("Gemini client initialized!")

Paste your Gemini API key: ··········
Gemini client initialized!


In [24]:
response = gemini_client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Explain media bias in two sentences."
)

print(response.text)

Media bias occurs when journalists and news outlets let personal opinions, political agendas, or corporate interests influence the selection and framing of stories. This practice distorts objective reality, leading to the presentation of news that subtly or overtly favors a specific viewpoint.


In [8]:

import pandas as pd

# Load the uploaded CSV.
df = pd.read_csv("bias_clean.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())


Dataset shape: (10832, 7)

Column names:
['url', 'topic', 'date', 'title', 'site', 'bias', 'page_text']

First 5 rows:


,url,topic,date,title,site,bias,page_text
0,https://www.npr.org/2021/01/01/952336030/what-...,general-news,2021-01-01,"What Got Us Through 2020? For Many, It Was Hob...",NPR (Online News),leaning-left,"What Got Us Through 2020? For Many, It Was Hob..."
1,https://www.cbsnews.com/news/new-years-eve-gat...,general-news,2020-12-31,New Year's Eve gatherings could accelerate COV...,CBS News (Online),leaning-left,New Year's Eve gatherings could accelerate COV...
2,https://nypost.com/2020/12/31/heres-how-countr...,general-news,2020-12-31,Here’s how countries around the world are ring...,New York Post (News),leaning-right,Here’s how countries around the world are ring...
3,https://www.theguardian.com/world/2020/dec/31/...,coronavirus,2020-12-31,Wisconsin health worker 'deliberately spoiled ...,The Guardian,left,Police in Wisconsin said on Thursday evening t...
4,https://www.foxnews.com/us/wis-hospital-intent...,coronavirus,2020-12-31,Wisconsin hospital employee 'intentionally' re...,Fox News Digital,right,A Wisconsin-based hospital on Wednesday said a...



Missing values:
url          0
topic        0
date         0
title        0
site         0
bias         0
page_text    0
dtype: int64


In [9]:

required_columns = {"title", "page_text", "bias"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {df.columns.tolist()}"
    )

# Fill missing titles and article text safely.
df["title"] = df["title"].fillna("").astype(str)
df["page_text"] = df["page_text"].fillna("").astype(str)

# Remove rows with no article text.
df = df[df["page_text"].str.strip().ne("")].copy()

# Preserve the original labels in a separate column.
df["source_bias"] = df["bias"].copy()

df.reset_index(drop=True, inplace=True)

print("Usable articles:", len(df))
print("\nOriginal source-bias distribution:")
print(df["source_bias"].value_counts(dropna=False))


Usable articles: 10832

Original source-bias distribution:
source_bias
right            2986
leaning-left     2951
left             1930
leaning-right    1487
center           1478
Name: count, dtype: int64


In [32]:
def detect_bias_llm(title, page_text, site):
    prompt = f"""
You are a media-source political orientation classification
analyst. Your task is to classify the general political
leaning of a news publication, NOT to determine whether
an individual article uses biased language.

NEWS SOURCE: {site}
ARTICLE TITLE: {title}
ARTICLE TEXT:
{str(page_text)[:6000]}

CLASSIFICATION TASK:
Classify the general political orientation of the news
source into exactly ONE of the following categories:

1. left
2. leaning-left
3. center
4. leaning-right
5. right

CATEGORY GUIDELINES:
- left: Generally associated with a distinctly left-wing
  editorial or political orientation.
- leaning-left: Generally associated with a moderate
  left-of-center editorial or political orientation.
- center: Generally associated with a centrist orientation
  or relatively balanced political positioning.
- leaning-right: Generally associated with a moderate
  right-of-center editorial or political orientation.
- right: Generally associated with a distinctly right-wing
  editorial or political orientation.

IMPORTANT RULES:
1. Classify the publication's general political leaning,
   not the bias of this specific article.
2. Do not automatically classify a source based on the
   political topic, the people mentioned, or one article's
   viewpoint.
3. Use the publication identity and any reliable evidence
   available in the supplied information.
4. A source can publish articles that do not reflect its
   general editorial orientation.
5. Do not invent facts about the publication or assume
   that all sources have a political leaning.
6. If the supplied information is insufficient to make a
   defensible classification, state that limitation.
7. Distinguish the source's political leaning from the
   accuracy or reliability of its reporting.

OUTPUT FORMAT:
Predicted category: [left / leaning-left / center /
leaning-right / right / uncertain]

Confidence: [High / Medium / Low]

Reasoning: Briefly explain the classification and the
evidence supporting it. Do not claim external verification
if none was performed.
"""

    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [34]:
test_row = df.iloc[0]

test_result = detect_bias_llm(
    test_row["title"],
    test_row["page_text"],
    test_row["site"]
)

print(test_result)

Predicted category: leaning-left

Confidence: High

Reasoning: NPR (National Public Radio) is widely recognized by independent media watchdogs, academic studies, and public consensus as having a moderate center-left editorial leaning, often categorized as "leaning-left." While this particular article is a human-interest piece about hobbies and coping mechanisms during the 2020 pandemic and contains no inherently partisan arguments, the classification is based on the general institutional profile and editorial tendencies of the news publication as a whole, rather than the content of this specific article.


In [37]:
import re
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# The five rows you tested
# test_df should contain the original dataset columns
# and the Gemini output column "llm_bias_analysis"

def extract_category(text):
    text = str(text).lower()

    match = re.search(
        r"predicted category\s*:\s*"
        r"(left|leaning-left|center|leaning-right|right)\b",
        text
    )

    return match.group(1) if match else None

test_df["predicted_bias"] = test_df["llm_bias_analysis"].apply(
    extract_category
)

# Exclude responses that did not produce a valid category
valid = test_df.dropna(subset=["predicted_bias"]).copy()

print("Valid predictions:", len(valid))
print("Original labels:", len(test_df))

if len(valid) > 0:
    accuracy = accuracy_score(
        valid["bias"].str.lower().str.strip(),
        valid["predicted_bias"]
    )

    print(f"\nAccuracy: {accuracy * 100:.2f}%")
    print("\nClassification report:")
    print(classification_report(
        valid["bias"].str.lower().str.strip(),
        valid["predicted_bias"],
        labels=[
            "left", "leaning-left", "center",
            "leaning-right", "right"
        ],
        zero_division=0
    ))

    display(valid[["title", "bias", "predicted_bias"]])

Valid predictions: 5
Original labels: 5

Accuracy: 80.00%

Classification report:
               precision    recall  f1-score   support

         left       0.00      0.00      0.00         1
 leaning-left       0.67      1.00      0.80         2
       center       0.00      0.00      0.00         0
leaning-right       1.00      1.00      1.00         1
        right       1.00      1.00      1.00         1

     accuracy                           0.80         5
    macro avg       0.53      0.60      0.56         5
 weighted avg       0.67      0.80      0.72         5



,title,bias,predicted_bias
0,"What Got Us Through 2020? For Many, It Was Hob...",leaning-left,leaning-left
1,New Year's Eve gatherings could accelerate COV...,leaning-left,leaning-left
2,Here’s how countries around the world are ring...,leaning-right,leaning-right
3,Wisconsin health worker 'deliberately spoiled ...,left,leaning-left
4,Wisconsin hospital employee 'intentionally' re...,right,right
